# 🔥 Introduzione a PyTorch

**Esercitazione di Reti Neurali** — Parte 1/3

> Questo notebook introduce i concetti fondamentali di PyTorch necessari per l'esercitazione pratica.
> Basato su: [PyTorch in One Hour](https://sebastianraschka.com/teaching/pytorch-1h/) di Sebastian Raschka.

⏱️ **Durata stimata:** 30–45 minuti

---

## Indice

1. Cos'è PyTorch e i suoi 3 componenti principali
2. Tensori: creazione, tipi, operazioni
3. Grafi di computazione
4. Differenziazione automatica (autograd)
5. Implementare una rete neurale con `nn.Module`
6. DataLoader e Dataset
7. Il training loop
8. Salvare e caricare modelli
9. Esecuzione su GPU


## 1. Cos'è PyTorch?

PyTorch è una libreria open-source per il deep learning basata su Python. È la libreria più usata nella ricerca (Papers With Code, 2019+).

I **tre componenti principali** sono:
1. **Tensor library** — struttura dati fondamentale (simile a NumPy, ma con supporto GPU)
2. **Autograd** — motore di differenziazione automatica per calcolare gradienti
3. **Deep learning utilities** — moduli, loss function, ottimizzatori, data loader

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_01.webp" width="700">


### Deep Learning nel contesto dell'AI

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_02.webp" width="500">

- **AI** → campo ampio, sistemi che svolgono compiti "intelligenti"
- **Machine Learning** → algoritmi che apprendono dai dati
- **Deep Learning** → reti neurali con molti layer nascosti

Il workflow tipico del supervised learning:

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_03.webp" width="600">


## 2. Tensori

I tensori generalizzano scalari, vettori e matrici a dimensioni arbitrarie.

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_06.webp" width="600">


In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.10.0+cu128
CUDA disponibile: True
GPU: Tesla T4


### 2.1 Creazione di tensori

In [2]:
# Scalare (0D), vettore (1D), matrice (2D), tensore 3D
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2], [3, 4]])
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

print("0D:", tensor0d, "  shape:", tensor0d.shape)
print("1D:", tensor1d, "  shape:", tensor1d.shape)
print("2D:\n", tensor2d, " shape:", tensor2d.shape)
print("3D:\n", tensor3d, " shape:", tensor3d.shape)

0D: tensor(1)   shape: torch.Size([])
1D: tensor([1, 2, 3])   shape: torch.Size([3])
2D:
 tensor([[1, 2],
        [3, 4]])  shape: torch.Size([2, 2])
3D:
 tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])  shape: torch.Size([2, 2, 2])


### 2.2 Tipi di dato

In [3]:
# Da interi Python -> int64
t_int = torch.tensor([1, 2, 3])
print(f"Intero: {t_int.dtype}")

# Da float Python -> float32 (default per il deep learning!)
t_float = torch.tensor([1.0, 2.0, 3.0])
print(f"Float: {t_float.dtype}")

# Conversione di tipo con .to()
t_converted = t_int.to(torch.float32)
print(f"Convertito: {t_converted.dtype}")

Intero: torch.int64
Float: torch.float32
Convertito: torch.float32


### 2.3 Operazioni comuni sui tensori

In [4]:
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])

print("Shape:", tensor2d.shape)
print()

# Reshape e View
print("Reshape(3,2):\n", tensor2d.reshape(3, 2))
print("View(3,2):\n", tensor2d.view(3, 2))
print()

# Trasposta
print("Trasposta:\n", tensor2d.T)
print()

# Prodotto matriciale: matmul o @
result = tensor2d @ tensor2d.T
print("Prodotto matriciale (A @ A^T):\n", result)

Shape: torch.Size([2, 3])

Reshape(3,2):
 tensor([[1, 2],
        [3, 4],
        [5, 6]])
View(3,2):
 tensor([[1, 2],
        [3, 4],
        [5, 6]])

Trasposta:
 tensor([[1, 4],
        [2, 5],
        [3, 6]])

Prodotto matriciale (A @ A^T):
 tensor([[14, 32],
        [32, 77]])


## 3. Grafi di computazione

Una rete neurale può essere rappresentata come un **grafo di computazione diretto**: ogni nodo è un'operazione, ogni arco trasporta tensori.

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_07.webp" width="600">

Esempio: forward pass di una regressione logistica.


In [5]:
import torch.nn.functional as F

y = torch.tensor([1.0])   # label vera
x1 = torch.tensor([1.1])  # feature di input
w1 = torch.tensor([2.2])  # peso
b = torch.tensor([0.0])   # bias

z = x1 * w1 + b           # net input
a = torch.sigmoid(z)      # attivazione (sigmoid)

loss = F.binary_cross_entropy(a, y)
print(f"Loss: {loss.item():.4f}")

Loss: 0.0852


## 4. Differenziazione automatica (Autograd)

PyTorch costruisce il grafo di computazione automaticamente per i tensori con `requires_grad=True`, e calcola i gradienti tramite **backpropagation** (regola della catena).

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_08.webp" width="600">


In [6]:
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)  # parametro trainable
b = torch.tensor([0.0], requires_grad=True)    # parametro trainable

z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

# Metodo 1: grad() esplicito
grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)
print(f"∂L/∂w1 = {grad_L_w1[0].item():.4f}")
print(f"∂L/∂b  = {grad_L_b[0].item():.4f}")

∂L/∂w1 = -0.0898
∂L/∂b  = -0.0817


In [7]:
# Metodo 2: .backward() — il più usato in pratica
# (ricalcoliamo perché il grafo è stato consumato)
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

loss.backward()  # calcola tutti i gradienti in un colpo

print(f"w1.grad = {w1.grad}")
print(f"b.grad  = {b.grad}")

w1.grad = tensor([-0.0898])
b.grad  = tensor([-0.0817])


## 5. Implementare una rete neurale

In PyTorch si definiscono modelli come sottoclassi di `torch.nn.Module`:
- `__init__`: si definiscono i layer
- `forward`: si definisce il flusso dei dati

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_09.webp" width="500">


In [8]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 30),  # 1° hidden layer
            torch.nn.ReLU(),
            torch.nn.Linear(30, 20),          # 2° hidden layer
            torch.nn.ReLU(),
            torch.nn.Linear(20, num_outputs),  # output layer
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

model = NeuralNetwork(50, 3)
print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParametri trainable totali: {num_params}")

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

Parametri trainable totali: 2213


In [9]:
# Forward pass con input casuale
torch.manual_seed(123)
X = torch.rand((1, 50))

# Con gradienti (per training)
out = model(X)
print("Con grad_fn:", out)

# Senza gradienti (per inferenza)
with torch.no_grad():
    out = model(X)
    probas = torch.softmax(out, dim=1)
    print("Probabilità:", probas)
    print("Somma:", probas.sum().item())

Con grad_fn: tensor([[0.1449, 0.0248, 0.1861]], grad_fn=<AddmmBackward0>)
Probabilità: tensor([[0.3414, 0.3028, 0.3558]])
Somma: 1.0


## 6. Dataset e DataLoader

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_10.webp" width="600">

- `Dataset`: definisce **come** caricare un singolo esempio
- `DataLoader`: gestisce **batching**, **shuffling**, **parallelismo**


In [10]:
from torch.utils.data import Dataset, DataLoader

# Dataset giocattolo
X_train = torch.tensor([
    [-1.2, 3.1], [-0.9, 2.9], [-0.5, 2.6],
    [2.3, -1.1], [2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([[-0.8, 2.8], [2.6, -1.6]])
y_test = torch.tensor([0, 1])

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

print(f"Training set: {len(train_ds)} esempi")
print(f"Test set: {len(test_ds)} esempi")

Training set: 5 esempi
Test set: 2 esempi


In [11]:
torch.manual_seed(123)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, drop_last=True)
test_loader = DataLoader(test_ds, batch_size=2, shuffle=False)

print("Iterazione sul training loader:")
for idx, (x, y) in enumerate(train_loader):
    print(f"  Batch {idx+1}: features shape={x.shape}, labels={y.tolist()}")

Iterazione sul training loader:
  Batch 1: features shape=torch.Size([2, 2]), labels=[1, 0]
  Batch 2: features shape=torch.Size([2, 2]), labels=[0, 0]


## 7. Il Training Loop

Mettiamo tutto insieme: modello, loss function, ottimizzatore, data loader.


In [12]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()  # modalità training
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()  # azzera i gradienti
        loss.backward()        # calcola i gradienti (backprop)
        optimizer.step()       # aggiorna i pesi

        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"Batch {batch_idx+1}/{len(train_loader)} | "
              f"Loss: {loss:.4f}")

model.eval()  # modalità valutazione

Epoch 01/3 | Batch 1/2 | Loss: 0.7487
Epoch 01/3 | Batch 2/2 | Loss: 0.6450
Epoch 02/3 | Batch 1/2 | Loss: 0.4423
Epoch 02/3 | Batch 2/2 | Loss: 0.1256
Epoch 03/3 | Batch 1/2 | Loss: 0.0269
Epoch 03/3 | Batch 2/2 | Loss: 0.0043


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)

In [13]:
# Valutazione
def compute_accuracy(model, dataloader):
    model.eval()
    correct = 0.0
    total = 0
    for features, labels in dataloader:
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum()
        total += len(labels)
    return (correct / total).item()

print(f"Train accuracy: {compute_accuracy(model, train_loader)*100:.1f}%")
print(f"Test accuracy:  {compute_accuracy(model, test_loader)*100:.1f}%")

Train accuracy: 100.0%
Test accuracy:  100.0%


## 8. Salvare e caricare modelli

In [14]:
# Salvataggio
torch.save(model.state_dict(), "model.pth")
print("Modello salvato!")

# Caricamento
model_loaded = NeuralNetwork(2, 2)
model_loaded.load_state_dict(torch.load("model.pth", weights_only=True))
print("Modello caricato!")

# Verifica
print(f"Accuracy modello caricato: {compute_accuracy(model_loaded, test_loader)*100:.1f}%")

Modello salvato!
Modello caricato!
Accuracy modello caricato: 100.0%


## 9. Esecuzione su GPU

Trasferire modello e dati sulla GPU è semplice:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
features, labels = features.to(device), labels.to(device)
```


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso: {device}")

# Tensori su GPU
if torch.cuda.is_available():
    t1 = torch.tensor([1., 2., 3.]).to(device)
    t2 = torch.tensor([4., 5., 6.]).to(device)
    print(f"Somma su GPU: {t1 + t2}")
    print(f"Device del risultato: {(t1+t2).device}")

Device in uso: cuda
Somma su GPU: tensor([5., 7., 9.], device='cuda:0')
Device del risultato: cuda:0


In [16]:
# Training completo su GPU
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

for epoch in range(3):
    model.train()
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

print(f"Training completato su {device}!")

# Nota: anche compute_accuracy va adattata per GPU
def compute_accuracy_gpu(model, dataloader, device):
    model.eval()
    correct = 0.0
    total = 0
    for features, labels in dataloader:
        features, labels = features.to(device), labels.to(device)
        with torch.no_grad():
            logits = model(features)
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum()
        total += len(labels)
    return (correct / total).item()

print(f"Train accuracy: {compute_accuracy_gpu(model, train_loader, device)*100:.1f}%")
print(f"Test accuracy:  {compute_accuracy_gpu(model, test_loader, device)*100:.1f}%")

Training completato su cuda!
Train accuracy: 100.0%
Test accuracy:  100.0%


---

## ✅ Riepilogo

| Concetto | PyTorch |
|----------|---------|
| Struttura dati | `torch.tensor` |
| Tipo di dato | `.dtype`, `.to(torch.float32)` |
| Gradienti | `requires_grad=True`, `.backward()` |
| Modello | sottoclasse di `nn.Module` |
| Dataset | sottoclasse di `Dataset` |
| Batching | `DataLoader` |
| Loss | `F.cross_entropy`, `F.binary_cross_entropy`, `F.mse_loss` |
| Ottimizzatore | `torch.optim.SGD`, `Adam`, ... |
| GPU | `.to("cuda")` |
| Salvataggio | `torch.save`, `torch.load` |

---

➡️ **Ora passiamo al Notebook 2: Esercizi pratici!**
